In [25]:
import os
from google.cloud import bigquery

project_id = os.environ.get("GCP_PROJECT", "cancer-multiomics")
client = bigquery.Client(project=project_id)

# When running locally, install compatibility shims for BQ Studio imports
try:
    from google.colab.sql import bigquery as _test
except ImportError:
    import bq_compat
    bq_compat.install(client)

# Cancer Multi-Omics EDA

This notebook explores publicly available cancer genomics, transcriptomics, and proteomics data hosted on [BigQuery by ISB-CGC](https://isb-cgc.appspot.com/). The goal is to understand the shape, scale, and join-ability of these datasets before building any multi-omic analysis pipelines.

**Data sources:**
- **TCGA** (The Cancer Genome Atlas) — somatic mutations, RNA-seq expression, clinical data across 33 cancer types
- **CPTAC** (Clinical Proteomic Tumor Analysis Consortium) — mass-spec proteomics and phosphoproteomics

## Part 1: TCGA — Mutations and gene expression

### What columns does the mutation table have?

Before writing any analysis queries, we need to understand the schema. The `INFORMATION_SCHEMA.COLUMNS` view gives us every column name in the TCGA somatic mutation table. Key things to look for: patient identifiers, gene symbols, mutation coordinates, and variant classification.

In [26]:
# sql_engine: bigquery
# output_variable: mutation_column_names
# start _sql
_sql = """
SELECT column_name
FROM `isb-cgc-bq.TCGA.INFORMATION_SCHEMA.COLUMNS`
WHERE table_name = 'masked_somatic_mutation_hg38_gdc_current'
ORDER BY ordinal_position
""" # end _sql
from google.colab.sql import bigquery as _bqsqlcell
mutation_column_names = _bqsqlcell.run(_sql)
mutation_column_names

C:\Users\antho\AppData\Roaming\Python\Python313\site-packages\google\cloud\bigquery\table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,column_name
0,project_short_name
1,case_barcode
2,primary_site
3,Hugo_Symbol
4,Entrez_Gene_Id
...,...
147,varscan2
148,sample_barcode_tumor
149,sample_barcode_normal
150,aliquot_barcode_tumor


### How many patients per cancer type (clinical data)?

Querying the clinical table gives us the number of patients per TCGA project. This is our denominator — it tells us the total cohort size for each cancer type, regardless of which assays were performed.

In [27]:
# sql_engine: bigquery
# output_variable: cases_by_cancer_type_TCGA
# start _sql
_sql = """
SELECT proj__name, proj__project_id, COUNT(*) AS patient_count
FROM `isb-cgc-bq.TCGA.clinical_gdc_current`
GROUP BY proj__name, proj__project_id
ORDER BY patient_count DESC
""" # end _sql
from google.colab.sql import bigquery as _bqsqlcell
cases_by_cancer_type_TCGA = _bqsqlcell.run(_sql)
cases_by_cancer_type_TCGA

C:\Users\antho\AppData\Roaming\Python\Python313\site-packages\google\cloud\bigquery\table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,proj__name,proj__project_id,patient_count
0,Breast Invasive Carcinoma,TCGA-BRCA,1098
1,Glioblastoma Multiforme,TCGA-GBM,617
2,Ovarian Serous Cystadenocarcinoma,TCGA-OV,608
3,Lung Adenocarcinoma,TCGA-LUAD,585
4,Uterine Corpus Endometrial Carcinoma,TCGA-UCEC,560
5,Kidney Renal Clear Cell Carcinoma,TCGA-KIRC,537
6,Head and Neck Squamous Cell Carcinoma,TCGA-HNSC,528
7,Brain Lower Grade Glioma,TCGA-LGG,516
8,Thyroid Carcinoma,TCGA-THCA,507
9,Lung Squamous Cell Carcinoma,TCGA-LUSC,504


### How many mutations per cancer type?

Now we look at the mutation table directly. Counting total rows and distinct patients per `project_short_name` tells us two things: how many patients have mutation data (not all clinical patients do), and the raw mutation counts per cancer type.

In [28]:
# sql_engine: bigquery
# output_variable: mutations_and_cases_by_cancer_type
# start _sql
_sql = """
SELECT project_short_name, COUNT(*) AS row_count, COUNT(DISTINCT case_barcode) AS patient_count
FROM `isb-cgc-bq.TCGA.masked_somatic_mutation_hg38_gdc_current`
GROUP BY project_short_name
ORDER BY patient_count DESC
""" # end _sql
from google.colab.sql import bigquery as _bqsqlcell
mutations_and_cases_by_cancer_type = _bqsqlcell.run(_sql)
mutations_and_cases_by_cancer_type

C:\Users\antho\AppData\Roaming\Python\Python313\site-packages\google\cloud\bigquery\table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,project_short_name,row_count,patient_count
0,TCGA-BRCA,89568,968
1,TCGA-LUAD,194729,557
2,TCGA-UCEC,626945,511
3,TCGA-LGG,32780,509
4,TCGA-HNSC,87967,508
5,TCGA-PRAD,24779,490
6,TCGA-THCA,5834,488
7,TCGA-LUSC,172826,485
8,TCGA-SKCM,353450,467
9,TCGA-STAD,183107,431


### Mutation burden varies dramatically across cancer types

Raw mutation counts are misleading because cohort sizes differ. Dividing total mutations by patient count gives the average mutation burden per patient. This is a critical biological variable — hypermutated cancers like melanoma (UV damage) and endometrial cancer (MSI-high) have 10–20x more mutations than thyroid or prostate cancers.

In [29]:
mutations_and_cases_by_cancer_type['avg_mut_per_patient'] = mutations_and_cases_by_cancer_type['row_count'] / mutations_and_cases_by_cancer_type['patient_count']
mutations_and_cases_by_cancer_type

,project_short_name,row_count,patient_count,avg_mut_per_patient
0,TCGA-BRCA,89568,968,92.528926
1,TCGA-LUAD,194729,557,349.603232
2,TCGA-UCEC,626945,511,1226.898239
3,TCGA-LGG,32780,509,64.400786
4,TCGA-HNSC,87967,508,173.163386
5,TCGA-PRAD,24779,490,50.569388
6,TCGA-THCA,5834,488,11.954918
7,TCGA-LUSC,172826,485,356.342268
8,TCGA-SKCM,353450,467,756.852248
9,TCGA-STAD,183107,431,424.842227


### RNA-seq table schema

Shifting from mutations to gene expression. The RNA-seq table has a fundamentally different structure — let's see what columns are available. We're especially interested in the quantification columns (TPM, FPKM, raw counts) and the sample identifiers.

In [30]:
# sql_engine: bigquery
# output_variable: RNA_seq_column_names
# start _sql
_sql = """
SELECT column_name
FROM `isb-cgc-bq.TCGA.INFORMATION_SCHEMA.COLUMNS`
WHERE table_name = 'RNAseq_hg38_gdc_current'
ORDER BY ordinal_position
""" # end _sql
from google.colab.sql import bigquery as _bqsqlcell
RNA_seq_column_names = _bqsqlcell.run(_sql)
RNA_seq_column_names

C:\Users\antho\AppData\Roaming\Python\Python313\site-packages\google\cloud\bigquery\table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,column_name
0,project_short_name
1,primary_site
2,case_barcode
3,sample_barcode
4,aliquot_barcode
5,gene_name
6,gene_type
7,Ensembl_gene_id
8,Ensembl_gene_id_v
9,unstranded


Barcode hierarchy:

*   case_barcode: AKA patient. Can have multiple samples taken.
*   sample_barcode: AKA tumor and matched normal tissue. Can have multiple aliquots.
*   aliquot_barcode: An aliquot is a physical portion sent to differnt labs.





### RNA-seq coverage by cancer type

How many patients have RNA-seq data per cancer type, and how large is the table? This helps us understand which TCGA projects have sufficient expression data for multi-omic analysis, and gives a sense of the BigQuery costs (expression tables are much larger than mutation tables due to their dense structure).

In [31]:
# sql_engine: bigquery
# output_variable: seq_and_patient_count_by_cancer_type
# start _sql
_sql = """
SELECT
  project_short_name,
  COUNT(*) AS row_count,
  COUNT(DISTINCT case_barcode) AS patient_count
FROM `isb-cgc-bq.TCGA.RNAseq_hg38_gdc_current`
GROUP BY project_short_name
ORDER BY patient_count DESC
""" # end _sql
from google.colab.sql import bigquery as _bqsqlcell
seq_and_patient_count_by_cancer_type = _bqsqlcell.run(_sql)
seq_and_patient_count_by_cancer_type

C:\Users\antho\AppData\Roaming\Python\Python313\site-packages\google\cloud\bigquery\table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,project_short_name,row_count,patient_count
0,TCGA-BRCA,74677384,1095
1,TCGA-UCEC,35731096,557
2,TCGA-KIRC,37247696,533
3,TCGA-HNSC,34335824,521
4,TCGA-LUAD,36398400,517
5,TCGA-LGG,32394576,516
6,TCGA-THCA,34699808,505
7,TCGA-LUSC,34093168,501
8,TCGA-PRAD,33607856,497
9,TCGA-SKCM,28694072,469


In [32]:
import plotly.express as px

fig = px.bar(
    seq_and_patient_count_by_cancer_type.sort_values('patient_count'),
    x='patient_count',
    y='project_short_name',
    orientation='h',
    title='RNA-seq Patient Count by Cancer Type (TCGA)',
    labels={'patient_count': 'Patients', 'project_short_name': 'Cancer Type'},
)
fig.update_layout(height=600, yaxis_title='', margin=dict(l=120))
fig.show()

In [33]:
seq_and_patient_count_by_cancer_type['avg'] = seq_and_patient_count_by_cancer_type['row_count'] / seq_and_patient_count_by_cancer_type['patient_count']
seq_and_patient_count_by_cancer_type

,project_short_name,row_count,patient_count,avg
0,TCGA-BRCA,74677384,1095,68198.524201
1,TCGA-UCEC,35731096,557,64149.184919
2,TCGA-KIRC,37247696,533,69883.106942
3,TCGA-HNSC,34335824,521,65903.692898
4,TCGA-LUAD,36398400,517,70403.094778
5,TCGA-LGG,32394576,516,62780.186047
6,TCGA-THCA,34699808,505,68712.491089
7,TCGA-LUSC,34093168,501,68050.235529
8,TCGA-PRAD,33607856,497,67621.440644
9,TCGA-SKCM,28694072,469,61181.390192


**Key difference: sparse vs. dense tables.** Mutation tables are *sparse* — only mutated genes get rows, so a patient with 100 mutations has 100 rows. Expression tables are *dense* — every gene gets a row for every sample. The ~60,000–70,000 rows per patient roughly matches the number of annotated genes in the human genome (protein-coding, lncRNA, pseudogenes, and other non-coding transcripts).

### What types of genes are measured?

RNA-seq quantifies far more than just protein-coding genes. This query breaks down the ~60k gene entries per patient by `gene_type` to see how many are protein-coding, lncRNA, pseudogenes, miRNA, etc. This matters when deciding which genes to include in downstream analyses — for most multi-omic work, we'll filter to protein-coding genes.

In [34]:
# sql_engine: bigquery
# output_variable: count_by_gene_type
# start _sql
_sql = """
SELECT gene_type, COUNT(DISTINCT gene_name) AS gene_count
FROM `isb-cgc-bq.TCGA.RNAseq_hg38_gdc_current`
GROUP BY gene_type
ORDER BY gene_count DESC
""" # end _sql
from google.colab.sql import bigquery as _bqsqlcell
count_by_gene_type = _bqsqlcell.run(_sql)
count_by_gene_type

C:\Users\antho\AppData\Roaming\Python\Python313\site-packages\google\cloud\bigquery\table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,gene_type,gene_count
0,protein_coding,19938
1,lncRNA,16882
2,processed_pseudogene,10160
3,unprocessed_pseudogene,2612
4,miRNA,1879
5,snRNA,1837
6,misc_RNA,1279
7,TEC,1057
8,transcribed_unprocessed_pseudogene,938
9,snoRNA,793


In [35]:
fig = px.pie(
    count_by_gene_type.head(10),
    values='gene_count',
    names='gene_type',
    title='Gene Types Measured in RNA-seq (Top 10)',
)
fig.update_traces(textposition='inside', textinfo='label+percent')
fig.update_layout(height=500)
fig.show()

## Part 2: CPTAC — Adding proteomics to the picture

TCGA provides genomics (mutations) and transcriptomics (RNA-seq), but not proteomics. The [Clinical Proteomic Tumor Analysis Consortium (CPTAC)](https://proteomics.cancer.gov/programs/cptac) fills that gap with mass-spectrometry-based protein and phosphoprotein quantification for many of the same cancer types. ISB-CGC hosts CPTAC data in BigQuery alongside TCGA, making multi-omic joins possible.

Let's explore what's available.

In [36]:
# sql_engine: bigquery
# output_variable: CPTAC_table_names
# start _sql
_sql = """
SELECT table_name
FROM `isb-cgc-bq.CPTAC.INFORMATION_SCHEMA.TABLES`
""" # end _sql
from google.colab.sql import bigquery as _bqsqlcell
CPTAC_table_names = _bqsqlcell.run(_sql)
CPTAC_table_names

C:\Users\antho\AppData\Roaming\Python\Python313\site-packages\google\cloud\bigquery\table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,table_name
0,quant_phosphoproteome_CPTAC_LSCC_discovery_stu...
1,clinical_CPTAC3_other_pdc_current
2,quant_proteome_CPTAC_GBM_discovery_study_pdc_c...
3,quant_phosphoproteome_PTRC_HGSOC_FFPE_validati...
4,quant_proteome_beat_AML_baseline_clinical_pdc_...
...,...
76,clinical_CPTAC2_other_pdc_current
77,quant_phosphoproteome_AML_ex_vivo_drug_respons...
78,quant_acetylome_CPTAC_LSCC_discovery_study_pdc...
79,quant_proteome_AML_ex_vivo_drug_response_prima...


### CPTAC clinical table schema

Before querying CPTAC clinical data, we need to know what columns are available — particularly patient identifiers and disease annotations that we'll use to join with the proteomics and genomics tables.

In [37]:
# sql_engine: bigquery
# output_variable: clinical_gdc_columns
# start _sql
_sql = """
SELECT column_name
FROM `isb-cgc-bq.CPTAC.INFORMATION_SCHEMA.COLUMNS`
WHERE table_name = 'clinical_gdc_current'
ORDER BY ordinal_position
""" # end _sql
from google.colab.sql import bigquery as _bqsqlcell
clinical_gdc_columns = _bqsqlcell.run(_sql)
clinical_gdc_columns

C:\Users\antho\AppData\Roaming\Python\Python313\site-packages\google\cloud\bigquery\table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,column_name
0,submitter_id
1,case_id
2,follow__count
3,diag__treat__count
4,primary_site
...,...
78,diag__path__created_datetime
79,diag__path__updated_datetime
80,state
81,created_datetime


### How many patients per CPTAC project?

A quick count of distinct `case_id` values per project tells us the overall scale of each CPTAC phase.

In [38]:
# sql_engine: bigquery
# output_variable: CPTAC_case_count_by_project
# start _sql
_sql = """
    SELECT proj__name, proj__project_id, COUNT(DISTINCT case_id) AS patient_count
    FROM `isb-cgc-bq.CPTAC.clinical_gdc_current`
    GROUP BY proj__name, proj__project_id
    ORDER BY patient_count DESC
""" # end _sql
from google.colab.sql import bigquery as _bqsqlcell
CPTAC_case_count_by_project = _bqsqlcell.run(_sql)
CPTAC_case_count_by_project

C:\Users\antho\AppData\Roaming\Python\Python313\site-packages\google\cloud\bigquery\table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,proj__name,proj__project_id,patient_count
0,"CPTAC-Brain, Head and Neck, Kidney, Lung, Panc...",CPTAC-3,1683
1,"CPTAC-Breast, Colon, Ovary",CPTAC-2,342


### CPTAC case counts by cancer type

CPTAC-2 and CPTAC-3 are umbrella projects that each cover several cancer types. Breaking down by `primary_site` and `disease_type` shows exactly which cancers are represented and at what sample sizes — important for knowing which cross-omic comparisons are statistically viable.

In [39]:
# sql_engine: bigquery
# output_variable: CPTAC_case_count_by_cancer_type
# start _sql
_sql = """
SELECT proj__project_id, primary_site, disease_type, COUNT(DISTINCT case_id) AS patient_count
FROM `isb-cgc-bq.CPTAC.clinical_gdc_current`
GROUP BY proj__project_id, primary_site, disease_type
ORDER BY proj__project_id, patient_count DESC
""" # end _sql
from google.colab.sql import bigquery as _bqsqlcell
CPTAC_case_count_by_cancer_type = _bqsqlcell.run(_sql)
CPTAC_case_count_by_cancer_type

C:\Users\antho\AppData\Roaming\Python\Python313\site-packages\google\cloud\bigquery\table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,proj__project_id,primary_site,disease_type,patient_count
0,CPTAC-2,Breast,Ductal and Lobular Neoplasms,115
1,CPTAC-2,Colon,Adenomas and Adenocarcinomas,105
2,CPTAC-2,Ovary,"Cystic, Mucinous and Serous Neoplasms",71
3,CPTAC-2,Other and unspecified female genital organs,"Cystic, Mucinous and Serous Neoplasms",21
4,CPTAC-2,Breast,None,16
5,CPTAC-2,Retroperitoneum and peritoneum,"Cystic, Mucinous and Serous Neoplasms",10
6,CPTAC-2,Breast,"Cystic, Mucinous and Serous Neoplasms",1
7,CPTAC-2,Breast,Squamous Cell Neoplasms,1
8,CPTAC-2,Rectum,Adenomas and Adenocarcinomas,1
9,CPTAC-2,Breast,Adenomas and Adenocarcinomas,1


### Do CPTAC-2 and CPTAC-3 tables share identifiers?

Different CPTAC phases used different proteomics pipelines. Before we can join mutation, expression, and protein data across phases, we need to check whether CPTAC-2 (prospective) and CPTAC-3 (discovery) tables share the same identifier columns (`case_id`, `sample_id`, `aliquot_id`, etc.).

In [40]:
# sql_engine: bigquery
# output_variable: CPTAC2_columns
# start _sql
_sql = """
-- What identifiers does a CPTAC-2 prospective table use?
SELECT column_name
FROM `isb-cgc-bq.CPTAC.INFORMATION_SCHEMA.COLUMNS`
WHERE table_name = 'quant_proteome_prospective_breast_BI_pdc_current'
ORDER BY ordinal_position
""" # end _sql
from google.colab.sql import bigquery as _bqsqlcell
CPTAC2_columns = _bqsqlcell.run(_sql)
CPTAC2_columns

C:\Users\antho\AppData\Roaming\Python\Python313\site-packages\google\cloud\bigquery\table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,column_name
0,case_id
1,sample_id
2,aliquot_id
3,aliquot_submitter_id
4,aliquot_run_metadata_id
5,study_name
6,protein_abundance_log2ratio
7,gene_id
8,gene_symbol
9,NCBI_gene_id


In [41]:
# sql_engine: bigquery
# output_variable: CPTAC3_columns
# start _sql
_sql = """
-- What identifiers does a CPTAC-3 discovery table use?
SELECT column_name
FROM `isb-cgc-bq.CPTAC.INFORMATION_SCHEMA.COLUMNS`
WHERE table_name = 'quant_proteome_CPTAC_UCEC_discovery_study_pdc_current'
ORDER BY ordinal_position
""" # end _sql
from google.colab.sql import bigquery as _bqsqlcell
CPTAC3_columns = _bqsqlcell.run(_sql)
CPTAC3_columns

C:\Users\antho\AppData\Roaming\Python\Python313\site-packages\google\cloud\bigquery\table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,column_name
0,case_id
1,sample_id
2,aliquot_id
3,aliquot_submitter_id
4,aliquot_run_metadata_id
5,study_name
6,protein_abundance_log2ratio
7,gene_id
8,gene_symbol
9,NCBI_gene_id


## Part 3: Verifying Join Keys Across TCGA and CPTAC

The fundamental question for multi-omics integration: **can we link the same patient across datasets?**

Both CPTAC-2 and CPTAC-3 tables use `case_id` as the patient identifier. TCGA uses `case_barcode`. We need to check:

1. **What do these identifiers look like?** Are they the same format (e.g., TCGA barcodes like `TCGA-A2-A0T2`)?
2. **Do CPTAC-2 prospective patients appear in TCGA?** They should — CPTAC-2 ran proteomics on the *same tumor samples*.
3. **Do CPTAC-3 discovery patients appear in TCGA?** They shouldn't — CPTAC-3 enrolled independent cohorts.
4. **Can we join CPTAC data with CPTAC's own genomic tables** (RNA-seq, mutations) for the self-contained route?

### What do patient identifiers look like in each dataset?

Sampling identifiers from each source side by side. The critical question: are they the same format?

- **TCGA** tables use human-readable barcodes like `TCGA-A2-A0T2`
- **CPTAC proteomics** tables use `case_id` which may be a GDC UUID (like `a1b2c3d4-...`)
- **CPTAC clinical** table has *both* `case_id` and `submitter_id` — if `submitter_id` contains the TCGA barcode, it's the bridge between systems
- **CPTAC RNAseq/mutations** (ISB-CGC curated) use `case_barcode` — likely barcodes

`id_1` = the primary identifier from each table. `id_2` = a secondary identifier that might reveal the barcode format.

In [42]:
# sql_engine: bigquery
# output_variable: sample_identifiers
# start _sql
_sql = """
-- Sample patient identifiers from each source to compare formats
-- TCGA tables use human-readable barcodes; CPTAC proteomics uses GDC UUIDs
(SELECT 'TCGA clinical' AS source, submitter_id AS id_1, '' AS id_2
 FROM `isb-cgc-bq.TCGA.clinical_gdc_current`
 LIMIT 5)

UNION ALL

(SELECT 'TCGA mutations' AS source, case_barcode AS id_1, '' AS id_2
 FROM `isb-cgc-bq.TCGA.masked_somatic_mutation_hg38_gdc_current`
 GROUP BY case_barcode
 LIMIT 5)

UNION ALL

-- CPTAC proteomics: case_id is a UUID, but aliquot_submitter_id may contain a barcode
(SELECT 'CPTAC-2 proteome (breast)' AS source, case_id AS id_1, aliquot_submitter_id AS id_2
 FROM `isb-cgc-bq.CPTAC.quant_proteome_prospective_breast_BI_pdc_current`
 GROUP BY case_id, aliquot_submitter_id
 LIMIT 5)

UNION ALL

(SELECT 'CPTAC-3 proteome (UCEC)' AS source, case_id AS id_1, aliquot_submitter_id AS id_2
 FROM `isb-cgc-bq.CPTAC.quant_proteome_CPTAC_UCEC_discovery_study_pdc_current`
 GROUP BY case_id, aliquot_submitter_id
 LIMIT 5)

UNION ALL

-- CPTAC clinical table has BOTH case_id (UUID) and submitter_id (barcode) — the bridge
(SELECT 'CPTAC clinical (bridge)' AS source, case_id AS id_1, submitter_id AS id_2
 FROM `isb-cgc-bq.CPTAC.clinical_gdc_current`
 WHERE proj__project_id = 'CPTAC-2'
 LIMIT 5)

UNION ALL

(SELECT 'CPTAC RNAseq' AS source, case_barcode AS id_1, '' AS id_2
 FROM `isb-cgc-bq.CPTAC.RNAseq_hg38_gdc_current`
 GROUP BY case_barcode
 LIMIT 5)
""" # end _sql
from google.colab.sql import bigquery as _bqsqlcell
sample_identifiers = _bqsqlcell.run(_sql)
sample_identifiers

C:\Users\antho\AppData\Roaming\Python\Python313\site-packages\google\cloud\bigquery\table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,source,id_1,id_2
0,CPTAC-3 proteome (UCEC),9e2da4f9-118a-11e9-afb9-0a9c39d33490,CPT0022850001
1,CPTAC-3 proteome (UCEC),c9bdcd45-118a-11e9-afb9-0a9c39d33490,CPT0076860003
2,CPTAC-3 proteome (UCEC),759378e9-118a-11e9-afb9-0a9c39d33490,CPT0064210003
3,CPTAC-3 proteome (UCEC),dff78dee-118a-11e9-afb9-0a9c39d33490,NX7
4,CPTAC-3 proteome (UCEC),d8ac4953-118a-11e9-afb9-0a9c39d33490,NX3
5,CPTAC clinical (bridge),59b8a334-51b4-46f6-868d-c420aaa0eafb,01BR043
6,CPTAC clinical (bridge),5978194e-dfd2-4666-b716-a0f7d6e36566,11BR050
7,CPTAC clinical (bridge),2bba44c1-8268-47ff-831a-f96e133c4058,18BR004
8,CPTAC clinical (bridge),4077ccb6-ff96-44c7-b77f-2a548e60de20,11BR019
9,CPTAC clinical (bridge),007df8da-3d8c-497d-b7d3-3a02596f9577,22BR005


### Strategy 1: Do CPTAC-2 prospective patients exist in TCGA?

CPTAC-2 performed proteomics on original TCGA tumor samples. If the identifiers match, we can do true same-patient joins: TCGA mutations + TCGA RNA-seq + CPTAC proteomics + CPTAC phosphoproteomics — all from the same tumor.

In [ ]:
# sql_engine: bigquery
# output_variable: cptac2_tcga_overlap
# start _sql
_sql = """
-- How many CPTAC-2 breast proteome patients also appear in TCGA clinical?
-- CPTAC proteomics uses UUID case_id; TCGA uses barcode submitter_id
-- The CPTAC clinical table bridges them: it has both case_id and submitter_id
WITH cptac2_patients AS (
    SELECT DISTINCT case_id
    FROM `isb-cgc-bq.CPTAC.quant_proteome_prospective_breast_BI_pdc_current`
),
-- Use CPTAC clinical to get the barcode (submitter_id) for each CPTAC case_id
cptac2_with_barcode AS (
    SELECT DISTINCT p.case_id, cc.submitter_id 
    FROM cptac2_patients p
    JOIN `isb-cgc-bq.CPTAC.clinical_gdc_current` cc USING (case_id)
),
tcga_patients AS (
    SELECT DISTINCT submitter_id
    FROM `isb-cgc-bq.TCGA.clinical_gdc_current`
    WHERE proj__project_id = 'TCGA-BRCA'
)
SELECT
    (SELECT COUNT(*) FROM cptac2_patients) AS cptac2_breast_patients,
    (SELECT COUNT(*) FROM tcga_patients) AS tcga_brca_patients,
    COUNT(*) AS patients_in_both
FROM cptac2_with_barcode cb
JOIN tcga_patients t ON cb.submitter_id = t.submitter_id
""" # end _sql
from google.colab.sql import bigquery as _bqsqlcell
cptac2_tcga_overlap = _bqsqlcell.run(_sql)
cptac2_tcga_overlap

C:\Users\antho\AppData\Roaming\Python\Python313\site-packages\google\cloud\bigquery\table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,cptac2_breast_patients,tcga_brca_patients,patients_in_both
0,126,1098,0


### Strategy 1 expanded: All three CPTAC-2 prospective cohorts vs TCGA

Check all three CPTAC-2 prospective cancer types (breast, colon, ovarian) for overlap with TCGA in a single query.

In [44]:
# sql_engine: bigquery
# output_variable: cptac2_all_overlap
# start _sql
_sql = """
-- CPTAC-2 prospective overlap with TCGA across all three cancer types
-- Bridge through CPTAC clinical table to convert UUID case_id → barcode submitter_id
WITH cptac2_patients AS (
    SELECT DISTINCT 'Breast' AS cancer_type, case_id
    FROM `isb-cgc-bq.CPTAC.quant_proteome_prospective_breast_BI_pdc_current`
    UNION ALL
    SELECT DISTINCT 'Colon', case_id
    FROM `isb-cgc-bq.CPTAC.quant_proteome_prospective_colon_PNNL_qeplus_pdc_current`
    UNION ALL
    SELECT DISTINCT 'Ovarian', case_id
    FROM `isb-cgc-bq.CPTAC.quant_proteome_prospective_ovarian_PNNL_qeplus_pdc_current`
),
-- Resolve CPTAC UUIDs to barcodes via the clinical table
cptac2_with_barcode AS (
    SELECT DISTINCT p.cancer_type, p.case_id, cc.submitter_id
    FROM cptac2_patients p
    JOIN `isb-cgc-bq.CPTAC.clinical_gdc_current` cc USING (case_id)
),
tcga_patients AS (
    SELECT DISTINCT
        CASE proj__project_id
            WHEN 'TCGA-BRCA' THEN 'Breast'
            WHEN 'TCGA-COAD' THEN 'Colon'
            WHEN 'TCGA-OV' THEN 'Ovarian'
        END AS cancer_type,
        submitter_id
    FROM `isb-cgc-bq.TCGA.clinical_gdc_current`
    WHERE proj__project_id IN ('TCGA-BRCA', 'TCGA-COAD', 'TCGA-OV')
)
SELECT
    c.cancer_type,
    COUNT(DISTINCT c.case_id) AS cptac2_patients,
    COUNT(DISTINCT t.submitter_id) AS tcga_patients,
    COUNT(DISTINCT CASE WHEN t.submitter_id IS NOT NULL THEN c.case_id END) AS overlap
FROM cptac2_with_barcode c
LEFT JOIN tcga_patients t
    ON c.submitter_id = t.submitter_id AND c.cancer_type = t.cancer_type
GROUP BY c.cancer_type
ORDER BY c.cancer_type
""" # end _sql
from google.colab.sql import bigquery as _bqsqlcell
cptac2_all_overlap = _bqsqlcell.run(_sql)
cptac2_all_overlap

C:\Users\antho\AppData\Roaming\Python\Python313\site-packages\google\cloud\bigquery\table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,cancer_type,cptac2_patients,tcga_patients,overlap


### Strategy 2: Can CPTAC-3 join with its own genomic tables?

CPTAC-3 discovery cohorts are independent from TCGA, but CPTAC has its own RNA-seq and mutation tables. If those use the same `case_barcode` / `case_id` identifiers as the proteomics tables, CPTAC-3 is fully self-contained for multi-omics analysis.

In [45]:
# sql_engine: bigquery
# output_variable: cptac3_internal_overlap
# start _sql
_sql = """
-- Can CPTAC-3 proteomics join with CPTAC's own RNA-seq and mutation data?
-- Test with UCEC (uterine) — one of the deepest CPTAC-3 cohorts
-- Bridge: CPTAC clinical table maps case_id (UUID) ↔ submitter_id (barcode)
WITH proteome_patients AS (
    SELECT DISTINCT case_id
    FROM `isb-cgc-bq.CPTAC.quant_proteome_CPTAC_UCEC_discovery_study_pdc_current`
),
-- Resolve proteome UUIDs to barcodes
proteome_with_barcode AS (
    SELECT DISTINCT p.case_id, cc.submitter_id
    FROM proteome_patients p
    JOIN `isb-cgc-bq.CPTAC.clinical_gdc_current` cc USING (case_id)
),
rnaseq_patients AS (
    SELECT DISTINCT case_barcode
    FROM `isb-cgc-bq.CPTAC.RNAseq_hg38_gdc_current`
),
mutation_patients AS (
    SELECT DISTINCT case_barcode
    FROM `isb-cgc-bq.CPTAC.masked_somatic_mutation_hg38_gdc_current`
)
SELECT
    (SELECT COUNT(*) FROM proteome_patients) AS ucec_proteome_patients,
    (SELECT COUNT(*) FROM rnaseq_patients) AS cptac_rnaseq_patients_all,
    (SELECT COUNT(*) FROM mutation_patients) AS cptac_mutation_patients_all,
    -- Proteome ↔ RNA-seq overlap (join on barcode)
    (SELECT COUNT(*)
     FROM proteome_with_barcode p
     JOIN rnaseq_patients r ON p.submitter_id = r.case_barcode
    ) AS proteome_rnaseq_overlap,
    -- Proteome ↔ Mutations overlap (join on barcode)
    (SELECT COUNT(*)
     FROM proteome_with_barcode p
     JOIN mutation_patients m ON p.submitter_id = m.case_barcode
    ) AS proteome_mutation_overlap
""" # end _sql
from google.colab.sql import bigquery as _bqsqlcell
cptac3_internal_overlap = _bqsqlcell.run(_sql)
cptac3_internal_overlap

C:\Users\antho\AppData\Roaming\Python\Python313\site-packages\google\cloud\bigquery\table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,ucec_proteome_patients,cptac_rnaseq_patients_all,cptac_mutation_patients_all,proteome_rnaseq_overlap,proteome_mutation_overlap
0,119,1682,1638,0,0


### Strategy 2 expanded: CPTAC-3 internal joins across all cancer types

Check how many CPTAC-3 proteome patients also have RNA-seq and mutation data within CPTAC's own genomic tables, broken down by cancer type. This tells us which CPTAC-3 cohorts are truly self-contained for multi-omics.

In [46]:
# sql_engine: bigquery
# output_variable: cptac3_coverage_by_type
# start _sql
_sql = """
-- CPTAC-3 multi-omics coverage: proteome patients who also have RNA-seq and mutations
-- Bridge: CPTAC clinical maps case_id (UUID) ↔ submitter_id (barcode)
WITH cptac_clinical AS (
    SELECT DISTINCT case_id, submitter_id, primary_site, disease_type
    FROM `isb-cgc-bq.CPTAC.clinical_gdc_current`
    WHERE proj__project_id = 'CPTAC-3'
),
rnaseq_patients AS (
    SELECT DISTINCT case_barcode
    FROM `isb-cgc-bq.CPTAC.RNAseq_hg38_gdc_current`
),
mutation_patients AS (
    SELECT DISTINCT case_barcode
    FROM `isb-cgc-bq.CPTAC.masked_somatic_mutation_hg38_gdc_current`
)
SELECT
    cc.primary_site,
    cc.disease_type,
    COUNT(DISTINCT cc.case_id) AS clinical_patients,
    COUNT(DISTINCT r.case_barcode) AS has_rnaseq,
    COUNT(DISTINCT m.case_barcode) AS has_mutations,
    COUNT(DISTINCT CASE
        WHEN r.case_barcode IS NOT NULL AND m.case_barcode IS NOT NULL
        THEN cc.case_id
    END) AS has_both_genomics
FROM cptac_clinical cc
LEFT JOIN rnaseq_patients r ON cc.submitter_id = r.case_barcode
LEFT JOIN mutation_patients m ON cc.submitter_id = m.case_barcode
GROUP BY cc.primary_site, cc.disease_type
ORDER BY clinical_patients DESC
""" # end _sql
from google.colab.sql import bigquery as _bqsqlcell
cptac3_coverage_by_type = _bqsqlcell.run(_sql)
cptac3_coverage_by_type

C:\Users\antho\AppData\Roaming\Python\Python313\site-packages\google\cloud\bigquery\table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,primary_site,disease_type,clinical_patients,has_rnaseq,has_mutations,has_both_genomics
0,Kidney,Adenomas and Adenocarcinomas,257,257,256,256
1,"Uterus, NOS",Adenomas and Adenocarcinomas,241,241,241,241
2,Bronchus and lung,Adenomas and Adenocarcinomas,230,230,229,229
3,Brain,Gliomas,211,210,199,198
4,Hematopoietic and reticuloendothelial systems,Myeloid Leukemias,173,0,0,0
5,Stomach,Adenomas and Adenocarcinomas,165,0,0,0
6,Pancreas,Ductal and Lobular Neoplasms,163,161,162,160
7,Bronchus and lung,Squamous Cell Neoplasms,109,109,109,109
8,Larynx,Squamous Cell Neoplasms,49,49,48,48
9,Other and unspecified parts of tongue,Squamous Cell Neoplasms,21,21,21,21


### Full picture: Which CPTAC-3 cancer types also have proteomics + phosphoproteomics?

The queries above checked genomics (RNA-seq, mutations). But the real power of CPTAC is the proteomic layers. This query checks which CPTAC-3 cancer types have patients across proteomics AND phosphoproteomics tables, alongside the genomic data.

In [47]:
# sql_engine: bigquery
# output_variable: cptac3_proteomic_coverage
# start _sql
_sql = """
-- Which CPTAC-3 cancer types have BOTH proteome and phosphoproteome tables?
-- Check by looking for table names matching each cancer type
SELECT
    table_name,
    CASE
        WHEN table_name LIKE '%UCEC%' THEN 'Uterine (UCEC)'
        WHEN table_name LIKE '%LUAD%' THEN 'Lung adeno (LUAD)'
        WHEN table_name LIKE '%CCRCC%' OR table_name LIKE '%ccRCC%' THEN 'Kidney (CCRCC)'
        WHEN table_name LIKE '%GBM%' OR table_name LIKE '%gbm%' THEN 'Brain (GBM)'
        WHEN table_name LIKE '%LSCC%' THEN 'Lung squam (LSCC)'
        WHEN table_name LIKE '%HNSCC%' THEN 'Head & Neck (HNSCC)'
        WHEN table_name LIKE '%PDA%' OR table_name LIKE '%pdac%' THEN 'Pancreas (PDA)'
        ELSE 'Other'
    END AS cancer_type,
    CASE
        WHEN table_name LIKE '%proteome%' AND table_name NOT LIKE '%phospho%' THEN 'proteome'
        WHEN table_name LIKE '%phospho%' THEN 'phosphoproteome'
        WHEN table_name LIKE '%acetyl%' THEN 'acetylome'
        WHEN table_name LIKE '%ubiquit%' THEN 'ubiquitylome'
        ELSE 'other'
    END AS modality
FROM `isb-cgc-bq.CPTAC.INFORMATION_SCHEMA.TABLES`
WHERE table_name LIKE '%discovery%'
    AND table_name LIKE '%quant%'
ORDER BY cancer_type, modality
""" # end _sql
from google.colab.sql import bigquery as _bqsqlcell
cptac3_proteomic_coverage = _bqsqlcell.run(_sql)
cptac3_proteomic_coverage

C:\Users\antho\AppData\Roaming\Python\Python313\site-packages\google\cloud\bigquery\table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,table_name,cancer_type,modality
0,quant_acetylome_CPTAC_GBM_discovery_study_pdc_...,Brain (GBM),acetylome
1,quant_phosphoproteome_CPTAC_GBM_discovery_stud...,Brain (GBM),phosphoproteome
2,quant_proteome_CPTAC_GBM_discovery_study_pdc_c...,Brain (GBM),proteome
3,quant_phosphoproteome_CPTAC_HNSCC_discovery_st...,Head & Neck (HNSCC),phosphoproteome
4,quant_proteome_CPTAC_HNSCC_discovery_study_pdc...,Head & Neck (HNSCC),proteome
5,quant_phosphoproteome_CPTAC_CCRCC_discovery_st...,Kidney (CCRCC),phosphoproteome
6,quant_proteome_CPTAC_CCRCC_discovery_study_pdc...,Kidney (CCRCC),proteome
7,quant_acetylome_CPTAC_LUAD_discovery_study_pdc...,Lung adeno (LUAD),acetylome
8,quant_phosphoproteome_CPTAC_LUAD_discovery_stu...,Lung adeno (LUAD),phosphoproteome
9,quant_proteome_CPTAC_LUAD_discovery_study_pdc_...,Lung adeno (LUAD),proteome
